1. Change column names:
   + dr.rename(): new_name = old_name
   + dr.rename_with() and lambda
   + Apply pandas method with dr.pipe()

<br>

2. Change row names (index):
   + dr.column_to_rownames()
   + combine dr.column_to_rownames() with dr.mutate()
   + dr.rownames_to_column() (dr.has_rownames() must be True)
   + Apply pandas method dr.pipe()

In [1]:
import datar.all as dr
from datar import f
import pandas as pd
from pathlib import Path

pd.set_option("display.width", 200)

# Get the path object pointing to the ``notebooks`` directory that contains *.csv files 
data_dir = next(Path("/home").rglob("*/notebooks/*.csv")).parent

In [3]:
tb_emp = dr.tibble(pd.read_csv(data_dir/"emp.csv"))

print(tb_emp)

       id      name    salary  start_date        dept
  <int64>     <str> <float64>       <str>       <str>
0       1      Rick    623.30  2012-01-01          IT
1       2       Dan    515.20  2013-09-23  Operations
2       3  Michelle    611.00  2014-11-15          IT
3       4      Ryan    729.00  2014-05-11          HR
4       5      Gary    843.25  2015-03-27     Finance
5       6      Nina    578.00  2013-05-21          IT
6       7     Simon    632.80  2013-07-30  Operations
7       8      Guru    722.50  2014-06-17     Finance


# <span style="color:#1E90FF">1. Change column names</span>

In [4]:
#################
## dr.rename() ##
#################
''' new_name = old_name'''

tb_renamed = (
    tb_emp
    >> dr.rename(emp_id = f.id, emp_name = f.name)
)

print(tb_renamed.head())

   emp_id  emp_name    salary  start_date        dept
  <int64>     <str> <float64>       <str>       <str>
0       1      Rick    623.30  2012-01-01          IT
1       2       Dan    515.20  2013-09-23  Operations
2       3  Michelle    611.00  2014-11-15          IT
3       4      Ryan    729.00  2014-05-11          HR
4       5      Gary    843.25  2015-03-27     Finance


In [5]:
######################
## dr.rename_with() ##
######################

#----
## Normal function
#----

tb_renamed2 = (
    tb_emp
    >> dr.rename_with(str.upper, f[f.salary, f.dept])
)

print(tb_renamed2.tail())

       id   name    SALARY  start_date        DEPT
  <int64>  <str> <float64>       <str>       <str>
3       4   Ryan    729.00  2014-05-11          HR
4       5   Gary    843.25  2015-03-27     Finance
5       6   Nina    578.00  2013-05-21          IT
6       7  Simon    632.80  2013-07-30  Operations
7       8   Guru    722.50  2014-06-17     Finance


In [6]:
#----
## Lambda function
#----

tb_pokemon = dr.tibble(
    pd.read_csv(data_dir/"pokemon.csv")
    >> dr.rename_with(lambda col: col.strip().replace(" ", "_").replace(".", ""))
)

print(tb_pokemon.head())

        #                   Name Type_1  Type_2   Total      HP  Attack  Defense  Sp_Atk  Sp_Def   Speed  Generation  Legendary
  <int64>                  <str>  <str>   <str> <int64> <int64> <int64>  <int64> <int64> <int64> <int64>     <int64>     <bool>
0       1              Bulbasaur  Grass  Poison     318      45      49       49      65      65      45           1      False
1       2                Ivysaur  Grass  Poison     405      60      62       63      80      80      60           1      False
2       3               Venusaur  Grass  Poison     525      80      82       83     100     100      80           1      False
3       3  VenusaurMega Venusaur  Grass  Poison     625      80     100      123     122     120      80           1      False
4       4             Charmander   Fire     NaN     309      39      52       43      60      50      65           1      False


In [7]:
###################
## pandas method ##
###################

from pipda import register_verb
dr.filter = register_verb(func=dr.filter_)

tb_renamed3 = (
    tb_emp
    >> dr.pipe(lambda f: f.add_prefix('emp_'))
    >> dr.filter(f.emp_salary > 600)
)

print(tb_renamed3.head())

   emp_id  emp_name  emp_salary emp_start_date    emp_dept
  <int64>     <str>   <float64>          <str>       <str>
0       1      Rick      623.30     2012-01-01          IT
2       3  Michelle      611.00     2014-11-15          IT
3       4      Ryan      729.00     2014-05-11          HR
4       5      Gary      843.25     2015-03-27     Finance
6       7     Simon      632.80     2013-07-30  Operations


In [8]:
'''--- Example with f.set_axis() ---'''

print(
    tb_emp
    >> dr.pipe(lambda f: f.set_axis(["ID", "Name", "Salary", "Start_Date", "Department"], axis=1))
)

       ID      Name    Salary  Start_Date  Department
  <int64>     <str> <float64>       <str>       <str>
0       1      Rick    623.30  2012-01-01          IT
1       2       Dan    515.20  2013-09-23  Operations
2       3  Michelle    611.00  2014-11-15          IT
3       4      Ryan    729.00  2014-05-11          HR
4       5      Gary    843.25  2015-03-27     Finance
5       6      Nina    578.00  2013-05-21          IT
6       7     Simon    632.80  2013-07-30  Operations
7       8      Guru    722.50  2014-06-17     Finance


# <span style="color:#1E90FF">2. Change row names</span>

In [9]:
#############################
## dr.column_to_rownames() ##
#############################

tb_to_rownames = (
    tb_emp
    >> dr.column_to_rownames('id')
)

print(tb_to_rownames.head())

       name    salary  start_date        dept
      <str> <float64>       <str>       <str>
1      Rick    623.30  2012-01-01          IT
2       Dan    515.20  2013-09-23  Operations
3  Michelle    611.00  2014-11-15          IT
4      Ryan    729.00  2014-05-11          HR
5      Gary    843.25  2015-03-27     Finance


In [10]:
######################################################
## Combine dr.column_to_rownames() with dr.mutate() ##
######################################################

print(
    tb_emp
    >> dr.select(~f.id)
    >> dr.mutate(emp_id = [f"employee_{i}" for i in range(1, len(tb_emp) + 1)])
    >> dr.column_to_rownames('emp_id')
)

                name    salary  start_date        dept
               <str> <float64>       <str>       <str>
employee_1      Rick    623.30  2012-01-01          IT
employee_2       Dan    515.20  2013-09-23  Operations
employee_3  Michelle    611.00  2014-11-15          IT
employee_4      Ryan    729.00  2014-05-11          HR
employee_5      Gary    843.25  2015-03-27     Finance
employee_6      Nina    578.00  2013-05-21          IT
employee_7     Simon    632.80  2013-07-30  Operations
employee_8      Guru    722.50  2014-06-17     Finance


In [11]:
#############################
## dr.rownames_to_column() ##
#############################

tb_from_rownames = (
    tb_to_rownames
    >> dr.rownames_to_column('emp_id')
)

print(tb_from_rownames.head())

       name emp_id    salary  start_date        dept
      <str>  <str> <float64>       <str>       <str>
0      Rick      1    623.30  2012-01-01          IT
1       Dan      2    515.20  2013-09-23  Operations
2  Michelle      3    611.00  2014-11-15          IT
3      Ryan      4    729.00  2014-05-11          HR
4      Gary      5    843.25  2015-03-27     Finance


In [12]:
###################
## pandas method ##
###################

from pipda import register_verb
dr.filter = register_verb(func = dr.filter_)


tb_set_index = (
    tb_emp
    >> dr.pipe(lambda f: f.set_index('id'))
    >> dr.filter( f.salary > 600)
)

print(tb_set_index.head())

        name    salary  start_date        dept
                                              
id     <str> <float64>       <str>       <str>
1       Rick    623.30  2012-01-01          IT
3   Michelle    611.00  2014-11-15          IT
4       Ryan    729.00  2014-05-11          HR
5       Gary    843.25  2015-03-27     Finance
7      Simon    632.80  2013-07-30  Operations
